## WSGI old

In [ ]:
import socket
from io import BytesIO
from urllib.parse import urlparse

In [ ]:
class WSGIRequest:
    def __init__(self, raw_data: bytes):
        self.raw_data = raw_data
        self.method = None
        self.path = None
        self.query_string = ""
        self.server_protocol = "HTTP/1.1"
        self.headers = {}
        self.body = b""

        self._parse()

    def _parse(self):
        header_part, _, body = self.raw_data.partition(b"\r\n\r\n")
        self.body = body

        lines = header_part.decode("utf-8", errors="replace").split("\r\n")
        request_line = lines[0]
        parts = request_line.split()

        if len(parts) != 3:
            raise ValueError("Некорректная строка запроса")

        self.method, raw_target, self.server_protocol = parts

        parsed = urlparse(raw_target)
        self.path = parsed.path
        self.query_string = parsed.query

        for line in lines[1:]:
            if ": " in line:
                key, value = line.split(": ", 1)
                self.headers[key.upper().replace("-", "_")] = value

       # print(self.headers)


class WSGIResponse:
    def __init__(self):
        self.status = "200 OK"
        self.headers = []
        self.body_chunks = []

    def start_response(self, status, headers, exc_info=None):
        self.status = status
        self.headers = headers

    def set_body(self, body_iterable):
        self.body_chunks = list(body_iterable)

    def to_http_bytes(self) -> bytes:
        body = b"".join(
            chunk if isinstance(chunk, bytes) else chunk.encode("utf-8")
            for chunk in self.body_chunks
        )

        headers = dict(self.headers)

        if "Content-Length" not in headers:
            headers["Content-Length"] = str(len(body))
        if "Content-Type" not in headers:
            headers["Content-Type"] = "text/plain; charset=utf-8"

        status_line = f"HTTP/1.1 {self.status}\r\n"
        header_lines = "".join(f"{k}: {v}\r\n" for k, v in headers.items())

        return (status_line + header_lines + "\r\n").encode("utf-8") + body

In [ ]:
class WSGIServer:
    def __init__(self, host="127.0.0.1", port=8080, application=None):
        self.host = host
        self.port = port
        self.application = application

    def build_environ(self, request: WSGIRequest):
        environ = {
            "REQUEST_METHOD": request.method,
            "PATH_INFO": request.path,
            "QUERY_STRING": request.query_string,
            "SERVER_NAME": self.host,
            "SERVER_PORT": str(self.port),
            "SERVER_PROTOCOL": request.server_protocol,
            "wsgi.version": (1, 0),
            "wsgi.url_scheme": "http",
            "wsgi.input": BytesIO(request.body),
            "wsgi.errors": BytesIO(),
            "wsgi.multithread": False,
            "wsgi.multiprocess": False,
            "wsgi.run_once": False,
        }

        for key, value in request.headers.items():
            if key == "CONTENT_TYPE":
                environ["CONTENT_TYPE"] = value
            elif key == "CONTENT_LENGTH":
                environ["CONTENT_LENGTH"] = value
            else:
                environ[f"HTTP_{key}"] = value

        return environ

    def handle_client(self, client_connection):
        raw_data = client_connection.recv(65536) # получаем данные через сокет в виде потока байтов

        try:
            request = WSGIRequest(raw_data) # обрабатываем запрос и парсим байты в headers
            print(request.headers)
            environ = self.build_environ(request) # на основе headers строим environ

            response = WSGIResponse() # делем ответ
            result = self.application(environ, response.start_response) # вызываем соответствующий метод приложения (app)
            response.set_body(result) # помещаем результат в body

            http_response = response.to_http_bytes() # собираем все в http формат и переводим в байты
            client_connection.sendall(http_response) # отправляем через сокет клиенту

            if hasattr(result, "close"):
                result.close()

        except Exception as e:
            error_body = f"Internal Server Error\n\n{e}".encode("utf-8")
            http_response = (
                b"HTTP/1.1 500 Internal Server Error\r\n"
                + b"Content-Type: text/plain; charset=utf-8\r\n"
                + f"Content-Length: {len(error_body)}\r\n".encode("utf-8")
                + b"\r\n"
                + error_body
            )
            client_connection.sendall(http_response)

        finally:
            client_connection.close()

    def serve_forever(self):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server_socket: # серверный слушающий сокет
            server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            server_socket.bind((self.host, self.port))
            server_socket.listen(5)

            print(f"WSGI server started on http://{self.host}:{self.port}")

            while True:
                client_connection, client_address = server_socket.accept() # устанавливаем tcp-соединение между сервером и клиентом
                # client_connetion - отдельный сокет для общения именно с этим клиентом
                # client_address - адрес клиента
                print(client_connection)
                print(f"Connection from {client_address}")
                self.handle_client(client_connection)

In [14]:
import socket
import sys
from io import BytesIO
from urllib.parse import urlparse

In [20]:
for line in BytesIO(b'A').readlines():
    print(line.decode())

A


## WSGI new

In [101]:
socket.AF_INET

<AddressFamily.AF_INET: 2>

In [102]:
socket.SOCK_STREAM

<SocketKind.SOCK_STREAM: 1>

In [86]:
class WSGIServer:
    def __init__(self, host="127.0.0.1", port=8080, application=None):
        self.host = host
        self.port = port # порт, на котором сервер будет слушать соединения
        self.application = application

    def serve_forever(self):
        # создаем TCP socket сервера
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server_socket:
            # разрешаем быстро перезапустить сервер на том же host:port
            server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    
            # привязываем сокет к адресу и порту
            server_socket.bind((self.host, self.port))
    
            # переводим сокет в режим прослушивания входящих подключений
            server_socket.listen(5)
    
            print(f"WSGI server started on http://{self.host}:{self.port}")
    
            # бесконечный цикл обработки клиентских подключений
            while True:
                # принимаем новое TCP-соединение от клиента
                client_connection, client_address = server_socket.accept()
                print(f'Client connection {client_connection}')
                print(f"Connection from {client_address}")

                # Здесь предполагаем, что один HTTP request -> один connection.
                # Так не всегда, можно также реализовать keep-alive HTTP.
                try:
                    # обрабатываем один HTTP request внутри этого соединения
                    self.handle_client(client_connection)
                finally:
                    # после обработки закрываем клиентское соединение
                    client_connection.close()

    def handle_client(self, client_connection):
        """
        Обрабатывает request от клиента:
            1. Читает request из сокета
            2. Парсит HTTP request
            3. Строит environ
            4. Реализует `start_response`, который сохраняет (status, response_headers) в `headers_set`
            5. Реализует `write`, который при первом вызове отправляет status line + headers в сокет,
               а затем отправляет body
        """
        try:
            raw_data = client_connection.recv(65536)  # читаем байты HTTP request из сокета
            if not raw_data:
                return
    
            method, path, query_string, server_protocol, headers, body = self.parse_request(raw_data)
            # парсим сырой HTTP request на method, path, headers, body
    
            environ = self.build_environ(
                method=method,
                path=path,
                query_string=query_string,
                server_protocol=server_protocol,
                headers=headers,
                body=body,
            )
            # строим WSGI environ на основе разобранного request
    
            headers_set = []
            response_started = False
    
            def write(data):
                nonlocal response_started
    
                if isinstance(data, str):
                    data = data.encode("utf-8")
    
                if not headers_set:
                    raise AssertionError("write() called before start_response()")
    
                if not response_started:
                    status, response_headers = headers_set
                    response_bytes = self.build_response_head(status, response_headers)
                    client_connection.sendall(response_bytes)
                    response_started = True
    
                if data:
                    client_connection.sendall(data)
    
            def start_response(status, response_headers, exc_info=None):
                """
                Сохраняет status и response_headers в headers_set.
                Сам по себе start_response ничего не отправляет в сокет.
                """
                if exc_info:
                    try:
                        if response_started:
                            raise exc_info[1].with_traceback(exc_info[2])
                    finally:
                        exc_info = None
                elif headers_set:
                    raise AssertionError("Headers already set")
    
                headers_set[:] = [status, response_headers]
    
            result = self.application(environ, start_response) # вызываем WSGI app
    
            try:
                for data in result:
                    if data:
                        write(data)
    
                if not response_started:
                    write(b"")  # если приложение не вернуло body, всё равно надо отправить headers
            finally:
                if hasattr(result, "close"):
                    result.close()
    
        except Exception as e:
            self.send_500_response(client_connection, e)

    def parse_request(self, raw_data: bytes):
        """Построчно разбирает байты из сокеты и выделяет из них headers, method, body и тд."""
        header_part, _, body = raw_data.partition(b"\r\n\r\n")

        lines = header_part.decode("utf-8", errors="replace").split("\r\n")
        if not lines or not lines[0]:
            raise ValueError("Empty request")

        request_line = lines[0]
        parts = request_line.split()
        if len(parts) != 3:
            raise ValueError("Invalid request line")

        method, raw_target, server_protocol = parts

        parsed = urlparse(raw_target)
        path = parsed.path or "/"
        query_string = parsed.query

        headers = {}
        for line in lines[1:]:
            if ": " in line:
                key, value = line.split(": ", 1)
                headers[key.upper().replace("-", "_")] = value

        return method, path, query_string, server_protocol, headers, body

    def build_environ(self, method, path, query_string, server_protocol, headers, body):
        """Строит environ на основе разорабнного request."""
        environ = {
            "REQUEST_METHOD": method,
            "PATH_INFO": path,
            "QUERY_STRING": query_string,
            "SERVER_NAME": self.host,
            "SERVER_PORT": str(self.port),
            "SERVER_PROTOCOL": server_protocol,
            "wsgi.version": (1, 0),
            "wsgi.url_scheme": "http",
            "wsgi.input": BytesIO(body), # file-like object, in this case the buffer of bytes
            "wsgi.errors": sys.stderr,
            "wsgi.multithread": False,
            "wsgi.multiprocess": False,
            "wsgi.run_once": False,
        }

        for key, value in headers.items():
            if key == "CONTENT_TYPE":
                environ["CONTENT_TYPE"] = value
            elif key == "CONTENT_LENGTH":
                environ["CONTENT_LENGTH"] = value
            else:
                environ[f"HTTP_{key}"] = value

        return environ

    def build_response_head(self, status, response_headers):
        headers_dict = dict(response_headers)

        if "Content-Type" not in headers_dict:
            headers_dict["Content-Type"] = "text/plain; charset=utf-8"

        status_line = f"HTTP/1.1 {status}\r\n"
        header_lines = "".join(f"{k}: {v}\r\n" for k, v in headers_dict.items())
        return (status_line + header_lines + "\r\n").encode("utf-8")

    def send_500_response(self, client_connection, error):
        error_body = f"Internal Server Error\n\n{error}".encode("utf-8")
        response = (
            b"HTTP/1.1 500 Internal Server Error\r\n"
            + b"Content-Type: text/plain; charset=utf-8\r\n"
            + f"Content-Length: {len(error_body)}\r\n".encode("utf-8")
            + b"\r\n"
            + error_body
        )
        client_connection.sendall(response)

In [87]:
def app(environ, start_response):
    path = environ["PATH_INFO"]
    method = environ["REQUEST_METHOD"]

    if path == "/echo" and method == "POST":
        body = environ["wsgi.input"].read()
        #print(body)

        status = "200 OK"
        headers = [
            ("Content-Type", "text/plain; charset=utf-8"),
            ("Content-Length", str(len(body))),
        ]
        start_response(status, headers)
        return [body]

    elif method == 'GET': 
        status = "200 OK"
        headers = [("Content-Type", "text/plain; charset=utf-8")]
        start_response(status, headers)
        return [b"Hello from GET method"]

In [88]:
wsgi_server = WSGIServer(host='127.0.0.1', port=8081, application=app)

In [90]:
wsgi_server.serve_forever() # curl -X POST http://127.0.0.1:8081/echo -d 'hello Zheka'

WSGI server started on http://127.0.0.1:8081
Client connection <socket.socket fd=116, family=2, type=1, proto=0, laddr=('127.0.0.1', 8081), raddr=('127.0.0.1', 42468)>
Connection from ('127.0.0.1', 42468)
Client connection <socket.socket fd=116, family=2, type=1, proto=0, laddr=('127.0.0.1', 8081), raddr=('127.0.0.1', 51752)>
Connection from ('127.0.0.1', 51752)
Client connection <socket.socket fd=116, family=2, type=1, proto=0, laddr=('127.0.0.1', 8081), raddr=('127.0.0.1', 53746)>
Connection from ('127.0.0.1', 53746)


KeyboardInterrupt: 

## ASGI

In [76]:
def outer():
    x = 10
    def inner():
        nonlocal x
        print(x)
        x = 1
    return inner

In [77]:
outer()()

10


In [100]:
x = 1
def f():
    global x
    x += 1
    print(x)
    return x
f()

2


2